In [1]:
import os
import sys
import json
import time
import random
import importlib.util
from pathlib import Path

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except TypeError:
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
except Exception:
    pass

print('Device:', device)
print('Seed:', SEED)
print('CUBLAS_WORKSPACE_CONFIG:', os.environ.get('CUBLAS_WORKSPACE_CONFIG'))


Device: cuda
Seed: 7
CUBLAS_WORKSPACE_CONFIG: :4096:8


In [2]:
DATA_ROOT = '../../Data/Tuco'
MAX_TRAIN = None
MAX_VAL = None

DOWNSAMPLE = True
WINDOW_RADIUS = 3
N_STACK = 2 * WINDOW_RADIUS + 1
CENTER_INDEX = WINDOW_RADIUS

ORIENTATIONS = [
    {'axis': 0, 'name': 'axial', 'target_hw': (112, 96)},
    {'axis': 1, 'name': 'coronal', 'target_hw': (96, 96)},
    {'axis': 2, 'name': 'sagittal', 'target_hw': (96, 112)},
] if DOWNSAMPLE else [
    {'axis': 0, 'name': 'axial', 'target_hw': (208, 192)},
    {'axis': 1, 'name': 'coronal', 'target_hw': (176, 192)},
    {'axis': 2, 'name': 'sagittal', 'target_hw': (176, 208)},
]

SEG_LABELS = [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28]
N_LABELS = len(SEG_LABELS)

BATCH_SIZE = 8
STEPS_PER_EPOCH = 200
VAL_STEPS = 50
EPOCHS = 1500
LEARNING_RATE = 1e-4
SMOOTH_WEIGHT = 0.1
MI_WEIGHT = 0.5
AUTO_RESUME = True
VAL_BATCH_SEED = SEED + 11
EVAL_PAIR_SEED = SEED + 29
MAX_EVAL_PAIRS = 2
EVAL_BATCH_SIZE = 16

RUN_NAME = '2p5d_pt_v2'
SAVE_DIR = '../trained_weights'
CKPT_DIR = os.path.join(SAVE_DIR, '2p5d_pt_v2_checkpoints')
BEST_MODEL_PATH = os.path.join(SAVE_DIR, '2p5d_dense_pt_v2_best.pth')
LATEST_TRAIN_STATE_PATH = os.path.join(CKPT_DIR, f'{RUN_NAME}_latest.pth')
EVAL_DIR = './artifacts/results/train_2p5d_v2'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print('Run name:', RUN_NAME)
print('Window radius:', WINDOW_RADIUS)
print('Native stack size:', N_STACK)
print('Orientations:', [(cfg['name'], cfg['target_hw']) for cfg in ORIENTATIONS])
print('Best model path:', BEST_MODEL_PATH)


Run name: 2p5d_pt_v2
Window radius: 3
Native stack size: 7
Orientations: [('axial', (112, 96)), ('coronal', (96, 96)), ('sagittal', (96, 112))]
Best model path: ../trained_weights/2p5d_dense_pt_v2_best.pth


In [3]:
import sys
from pathlib import Path

for _candidate in [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / 'Voxelmorph']:
    if (_candidate / 'tuco_dataset.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError('Could not find tuco_dataset.py from the current working directory')

from tuco_dataset import TucoDataset


def normalize_volume_contract(volume, norm_counts=None):
    arr = volume.astype(np.float32)
    arr_min = float(arr.min())
    arr_max = float(arr.max())

    if arr_max <= arr_min:
        mode = 'constant_zero'
        out = np.zeros_like(arr, dtype=np.float32)
    elif arr_min >= -1.001 and arr_max <= 1.001:
        if arr_min >= -1e-4 and arr_max <= 1.0001:
            mode = 'zero_one_to_minus_one_one'
            out = (2.0 * arr - 1.0).astype(np.float32)
        else:
            mode = 'already_-1_1'
            out = arr
    else:
        mode = 'minmax_to_minus_one_one'
        out = (2.0 * (arr - arr_min) / (arr_max - arr_min) - 1.0).astype(np.float32)

    if norm_counts is not None:
        norm_counts[mode] = norm_counts.get(mode, 0) + 1
    return out


def load_volumes(split, max_n):
    ds = TucoDataset(DATA_ROOT, split=split)
    if max_n is not None:
        from torch.utils.data import Subset
        ds = Subset(ds, range(min(max_n, len(ds))))

    moving_vols, fixed_vols = [], []
    moving_segs, fixed_segs = [], []
    norm_counts = {}

    for sample in ds:
        moving = sample['moving'].squeeze(0).numpy().astype(np.float32)
        fixed = sample['fixed'].squeeze(0).numpy().astype(np.float32)
        moving_seg = sample['moving_seg'].squeeze(0).numpy().astype(np.int16)
        fixed_seg = sample['fixed_seg'].squeeze(0).numpy().astype(np.int16)

        moving_vols.append(normalize_volume_contract(moving, norm_counts))
        fixed_vols.append(normalize_volume_contract(fixed, norm_counts))
        moving_segs.append(moving_seg)
        fixed_segs.append(fixed_seg)

    return moving_vols, fixed_vols, moving_segs, fixed_segs, norm_counts


print('Loading train volumes...')
train_mr_vols, train_ct_vols, train_mr_segs, train_ct_segs, train_norm_counts = load_volumes('train', MAX_TRAIN)
print('Loading val volumes...')
val_mr_vols, val_ct_vols, val_mr_segs, val_ct_segs, val_norm_counts = load_volumes('val', MAX_VAL)

sample_shape = tuple(train_mr_vols[0].shape)
stacks_per_subj = sum(sample_shape[cfg['axis']] - 2 * WINDOW_RADIUS for cfg in ORIENTATIONS)

print('Train volumes:', len(train_mr_vols))
print('Val volumes:', len(val_mr_vols))
print('Sample shape:', sample_shape)
print('Approx stacks per subject:', stacks_per_subj)
print('Train normalization modes:', train_norm_counts)
print('Val normalization modes:', val_norm_counts)
print('Value range example:', float(train_mr_vols[0].min()), float(train_mr_vols[0].max()))


Loading train volumes...
Found 144 volume pairs for split 'train'
Loading val volumes...
Found 18 volume pairs for split 'val'


KeyboardInterrupt: 

In [ ]:
def extract_slice(volume, axis, z):
    if axis == 0:
        return volume[z]
    if axis == 1:
        return volume[:, z, :]
    return volume[:, :, z]


def extract_stack(volume, axis, z, window_radius=WINDOW_RADIUS):
    wr = window_radius
    if axis == 0:
        return volume[z - wr:z + wr + 1]
    if axis == 1:
        return volume[:, z - wr:z + wr + 1, :].transpose(1, 0, 2)
    return volume[:, :, z - wr:z + wr + 1].transpose(2, 0, 1)


def resize_slice(slice_2d, target_hw, is_seg=False):
    target_h, target_w = target_hw
    interp = cv2.INTER_NEAREST if is_seg else cv2.INTER_LINEAR
    return cv2.resize(slice_2d, (target_w, target_h), interpolation=interp)


def resize_stack(stack, target_hw):
    resized = [resize_slice(stack[i], target_hw, is_seg=False) for i in range(stack.shape[0])]
    return np.ascontiguousarray(np.stack(resized).astype(np.float32))


def sample_cross_subject_pairs(n_subj, batch_size, rng):
    moving_ids = rng.integers(0, n_subj, size=batch_size)
    fixed_ids = rng.integers(0, n_subj, size=batch_size)
    for i in range(batch_size):
        while fixed_ids[i] == moving_ids[i]:
            fixed_ids[i] = rng.integers(0, n_subj)
    return moving_ids, fixed_ids


def build_batch(mr_vols, ct_vols, mr_segs, ct_segs, batch_size, orient_cfg, rng):
    n_subj = len(mr_vols)
    axis = orient_cfg['axis']
    target_hw = orient_cfg['target_hw']
    moving_ids, fixed_ids = sample_cross_subject_pairs(n_subj, batch_size, rng)

    moving_batch = []
    fixed_batch = []
    moving_seg_batch = []
    fixed_seg_batch = []
    z_batch = []

    for m_idx, f_idx in zip(moving_ids, fixed_ids):
        valid_depth = min(mr_vols[m_idx].shape[axis], ct_vols[f_idx].shape[axis])
        if valid_depth <= 2 * WINDOW_RADIUS:
            raise ValueError(f'Volume too shallow for axis {axis}: depth={valid_depth}')

        z = int(rng.integers(WINDOW_RADIUS, valid_depth - WINDOW_RADIUS))
        moving_stack = resize_stack(extract_stack(mr_vols[m_idx], axis, z), target_hw)
        fixed_stack = resize_stack(extract_stack(ct_vols[f_idx], axis, z), target_hw)
        moving_seg = resize_slice(extract_slice(mr_segs[m_idx], axis, z), target_hw, is_seg=True).astype(np.int64)
        fixed_seg = resize_slice(extract_slice(ct_segs[f_idx], axis, z), target_hw, is_seg=True).astype(np.int64)

        moving_batch.append(moving_stack)
        fixed_batch.append(fixed_stack)
        moving_seg_batch.append(moving_seg)
        fixed_seg_batch.append(fixed_seg)
        z_batch.append(z)

    return {
        'moving': np.ascontiguousarray(np.stack(moving_batch).astype(np.float32)),
        'fixed': np.ascontiguousarray(np.stack(fixed_batch).astype(np.float32)),
        'moving_seg': np.ascontiguousarray(np.stack(moving_seg_batch).astype(np.int64)),
        'fixed_seg': np.ascontiguousarray(np.stack(fixed_seg_batch).astype(np.int64)),
        'meta': {
            'orientation': orient_cfg['name'],
            'axis': axis,
            'target_hw': tuple(target_hw),
            'moving_ids': moving_ids.tolist(),
            'fixed_ids': fixed_ids.tolist(),
            'z_indices': z_batch,
        }
    }


def multi_orient_generator(mr_vols, ct_vols, mr_segs, ct_segs, batch_size=BATCH_SIZE, seed=None):
    rng = np.random.default_rng(seed)
    while True:
        orient_cfg = ORIENTATIONS[int(rng.integers(0, len(ORIENTATIONS)))]
        yield build_batch(mr_vols, ct_vols, mr_segs, ct_segs, batch_size, orient_cfg, rng)


def build_fixed_val_batches(n_batches=VAL_STEPS, batch_size=BATCH_SIZE, seed=VAL_BATCH_SEED):
    rng = np.random.default_rng(seed)
    batches = []
    for _ in range(n_batches):
        orient_cfg = ORIENTATIONS[int(rng.integers(0, len(ORIENTATIONS)))]
        batches.append(build_batch(val_mr_vols, val_ct_vols, val_mr_segs, val_ct_segs, batch_size, orient_cfg, rng))
    return batches


test_gen = multi_orient_generator(train_mr_vols, train_ct_vols, train_mr_segs, train_ct_segs, batch_size=2, seed=SEED)
test_batch = next(test_gen)
print('Orientation:', test_batch['meta']['orientation'])
print('Moving batch shape:', test_batch['moving'].shape)
print('Fixed batch shape:', test_batch['fixed'].shape)
print('Moving seg shape:', test_batch['moving_seg'].shape)
del test_gen, test_batch


Orientation: sagittal
Moving batch shape: (2, 7, 96, 112)
Fixed batch shape: (2, 7, 96, 112)
Moving seg shape: (2, 96, 112)


In [ ]:
def spatial_transform_2d(src, flow, mode='bilinear'):
    b, c, h, w = src.shape
    yy, xx = torch.meshgrid(
        torch.arange(h, device=src.device),
        torch.arange(w, device=src.device),
        indexing='ij'
    )
    base = torch.stack((xx, yy), dim=0).float().unsqueeze(0).expand(b, -1, -1, -1)
    new_locs = base + flow
    shape = torch.tensor([w - 1, h - 1], device=src.device).float().view(1, 2, 1, 1)
    new_locs = (new_locs / shape) * 2.0 - 1.0
    new_locs = new_locs.permute(0, 2, 3, 1)
    return F.grid_sample(src, new_locs, align_corners=True, mode=mode, padding_mode='border')


def seg_to_onehot(seg, labels=SEG_LABELS):
    return torch.stack([(seg == lbl) for lbl in labels], dim=1).float().to(seg.device)


def onehot_to_seg_numpy(seg_oh, labels=SEG_LABELS, threshold=0.5):
    seg_np = seg_oh.detach().cpu().numpy()
    out = np.zeros((seg_np.shape[0], seg_np.shape[2], seg_np.shape[3]), dtype=np.int16)
    for li, lbl in enumerate(labels):
        out[seg_np[:, li] > threshold] = lbl
    return out


def flow_gradient_loss_2d(flow):
    dy = flow[:, :, 1:, :] - flow[:, :, :-1, :]
    dx = flow[:, :, :, 1:] - flow[:, :, :, :-1]
    return dx.pow(2).mean() + dy.pow(2).mean()


def dice_loss(pred_onehot, gt_onehot, eps=1e-5):
    inter = torch.sum(pred_onehot * gt_onehot, dim=(0, 2, 3))
    union = torch.sum(pred_onehot, dim=(0, 2, 3)) + torch.sum(gt_onehot, dim=(0, 2, 3))
    return 1.0 - torch.mean((2.0 * inter + eps) / (union + eps))


class PatchwiseMI(nn.Module):
    def __init__(self, num_bins=32, min_clip=-1.0, max_clip=1.0, sigma_ratio=0.5):
        super().__init__()
        self.num_bins = int(num_bins)
        self.min_clip = float(min_clip)
        self.max_clip = float(max_clip)
        bin_centers = torch.linspace(self.min_clip, self.max_clip, self.num_bins)
        self.register_buffer('bin_centers', bin_centers)
        sigma = ((self.max_clip - self.min_clip) / max(self.num_bins - 1, 1)) * sigma_ratio
        self.preterm = 1.0 / (2 * (sigma ** 2) + 1e-8)

    def _soft_encoding(self, vol):
        vol = torch.clamp(vol.float(), self.min_clip, self.max_clip).unsqueeze(-1)
        diff = vol - self.bin_centers.view(1, 1, 1, 1, -1)
        enc = torch.exp(-self.preterm * diff.pow(2))
        enc = enc / (enc.sum(dim=-1, keepdim=True) + 1e-5)
        return enc

    def forward(self, moving, fixed):
        moving_enc = self._soft_encoding(moving)
        fixed_enc = self._soft_encoding(fixed)
        batch_size = moving.shape[0]
        moving_enc = moving_enc.view(batch_size, -1, self.num_bins)
        fixed_enc = fixed_enc.view(batch_size, -1, self.num_bins)
        joint = torch.bmm(moving_enc.transpose(1, 2), fixed_enc) / moving_enc.shape[1]
        pm = moving_enc.mean(dim=1, keepdim=True)
        pf = fixed_enc.mean(dim=1, keepdim=True)
        indep = torch.bmm(pm.transpose(1, 2), pf) + 1e-5
        mi = (joint * torch.log((joint + 1e-5) / indep)).sum(dim=(1, 2))
        return -mi.mean()


patchwise_mi = PatchwiseMI().to(device)


In [ ]:
class Vxm2p5dDenseCoreV2(nn.Module):
    def __init__(self, n_stack=N_STACK, enc_feats=(16, 32, 32, 32), final_feats=(32, 16), flow_scale=0.1):
        super().__init__()
        self.enc_blocks = nn.ModuleList()
        self.down_blocks = nn.ModuleList()
        in_ch = n_stack * 2

        for nf in enc_feats:
            self.enc_blocks.append(nn.Sequential(
                nn.Conv2d(in_ch, nf, 3, padding=1, bias=False),
                nn.BatchNorm2d(nf),
                nn.LeakyReLU(0.1),
            ))
            self.down_blocks.append(nn.Sequential(
                nn.Conv2d(nf, nf, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(nf),
                nn.LeakyReLU(0.1),
            ))
            in_ch = nf

        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),
        )

        self.up_blocks = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()

        in_ch = 32
        for skip_ch in reversed(enc_feats):
            self.up_blocks.append(nn.Upsample(scale_factor=2, mode='nearest'))
            self.dec_blocks.append(nn.Sequential(
                nn.Conv2d(in_ch + skip_ch, skip_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(skip_ch),
                nn.LeakyReLU(0.1),
            ))
            in_ch = skip_ch

        self.final_conv0 = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),
        )
        self.final_conv1 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),
        )

        self.flow_unscaled = nn.Conv2d(16, 2, 3, padding=1)
        nn.init.zeros_(self.flow_unscaled.weight)
        nn.init.zeros_(self.flow_unscaled.bias)

        self.flow_scale = nn.Conv2d(2, 2, 1, bias=False)
        self.flow_scale.weight.data.zero_()
        self.flow_scale.weight.data[0, 0, 0, 0] = flow_scale
        self.flow_scale.weight.data[1, 1, 0, 0] = flow_scale
        self.flow_scale.weight.requires_grad = False

    def forward(self, moving_stack, fixed_stack):
        x = torch.cat([moving_stack, fixed_stack], dim=1)
        skips = []
        for enc, ds in zip(self.enc_blocks, self.down_blocks):
            x = enc(x)
            skips.append(x)
            x = ds(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.up_blocks, self.dec_blocks, reversed(skips)):
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        x = self.final_conv0(x)
        x = self.final_conv1(x)
        return self.flow_scale(self.flow_unscaled(x))


model = Vxm2p5dDenseCoreV2().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print('First conv in_channels:', model.enc_blocks[0][0].in_channels)


Vxm2p5dDenseCoreV2(
  (enc_blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(14, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.1)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.1)
    )
    (2-3): 2 x Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.1)
    )
  )
  (down_blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=

In [ ]:
train_gen = multi_orient_generator(
    train_mr_vols,
    train_ct_vols,
    train_mr_segs,
    train_ct_segs,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
val_batches = build_fixed_val_batches()

train_history = {'total': [], 'dice': [], 'mi_loss': [], 'smooth': [], 'flow_abs_mean': [], 'flow_std': []}
val_history = {'total': [], 'dice': [], 'mi_loss': [], 'smooth': [], 'flow_abs_mean': [], 'flow_std': []}
best_loss = float('inf')


def build_arch_signature(model):
    state = model.state_dict()
    return {
        'model_class': model.__class__.__name__,
        'n_stack': N_STACK,
        'state_keys': list(state.keys()),
        'state_shapes': {k: tuple(v.shape) for k, v in state.items()},
        'state_dtypes': {k: str(v.dtype) for k, v in state.items()},
    }


def checkpoint_epoch_from_name(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    if '_epoch_' not in stem:
        return -1
    try:
        return int(stem.rsplit('_epoch_', 1)[1])
    except ValueError:
        return -1


def save_training_state(path, epoch_completed, epoch_train, epoch_val):
    ckpt = {
        'epoch': int(epoch_completed),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_history': train_history,
        'val_history': val_history,
        'train_metrics': epoch_train,
        'val_metrics': epoch_val,
        'best_loss': float(best_loss),
        'arch_signature': ARCH_SIGNATURE,
        'run_name': RUN_NAME,
    }
    torch.save(ckpt, path)


def load_checkpoint_candidate(path):
    try:
        ckpt = torch.load(path, map_location=device)
    except Exception as exc:
        print(f'Skipping {path}: load failed ({exc})')
        return None

    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        saved_signature = ckpt.get('arch_signature')
        if saved_signature is not None and saved_signature != ARCH_SIGNATURE:
            print(f'Skipping {path}: architecture mismatch')
            return None
        try:
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        except Exception as exc:
            print(f'Skipping {path}: restore failed ({exc})')
            return None
        return {
            'start_epoch': int(ckpt.get('epoch', 0)),
            'train_history': ckpt.get('train_history', train_history),
            'val_history': ckpt.get('val_history', val_history),
            'best_loss': float(ckpt.get('best_loss', float('inf'))),
            'path': path,
            'mode': 'full',
        }

    current_state = model.state_dict()
    if isinstance(ckpt, dict) and ckpt and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        if set(ckpt.keys()) != set(current_state.keys()):
            print(f'Skipping {path}: legacy weights do not match current model')
            return None
        try:
            model.load_state_dict(ckpt)
        except Exception as exc:
            print(f'Skipping {path}: legacy weights failed to load ({exc})')
            return None
        return {
            'start_epoch': max(checkpoint_epoch_from_name(path), 0),
            'train_history': {'total': [], 'dice': [], 'mi_loss': [], 'smooth': [], 'flow_abs_mean': [], 'flow_std': []},
            'val_history': {'total': [], 'dice': [], 'mi_loss': [], 'smooth': [], 'flow_abs_mean': [], 'flow_std': []},
            'best_loss': float('inf'),
            'path': path,
            'mode': 'weights_only',
        }

    print(f'Skipping {path}: unsupported checkpoint format')
    return None


def maybe_resume_training():
    candidates = []
    if os.path.exists(LATEST_TRAIN_STATE_PATH):
        candidates.append(LATEST_TRAIN_STATE_PATH)
    snapshot_paths = sorted(
        [os.path.join(CKPT_DIR, name) for name in os.listdir(CKPT_DIR) if name.startswith(f'{RUN_NAME}_epoch_') and name.endswith('.pth')],
        key=checkpoint_epoch_from_name,
        reverse=True,
    )
    candidates.extend(snapshot_paths)

    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        restored = load_checkpoint_candidate(path)
        if restored is not None:
            return restored
    return None


run_config = {
    'run_name': RUN_NAME,
    'seed': SEED,
    'data_root': DATA_ROOT,
    'max_train': MAX_TRAIN,
    'max_val': MAX_VAL,
    'downsample': DOWNSAMPLE,
    'window_radius': WINDOW_RADIUS,
    'n_stack': N_STACK,
    'orientations': ORIENTATIONS,
    'batch_size': BATCH_SIZE,
    'steps_per_epoch': STEPS_PER_EPOCH,
    'val_steps': VAL_STEPS,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'smooth_weight': SMOOTH_WEIGHT,
    'mi_weight': MI_WEIGHT,
    'seg_labels': SEG_LABELS,
    'auto_resume': AUTO_RESUME,
    'latest_train_state_path': LATEST_TRAIN_STATE_PATH,
}

with open(os.path.join(EVAL_DIR, 'run_config.json'), 'w') as f:
    json.dump(run_config, f, indent=2)

ARCH_SIGNATURE = build_arch_signature(model)
start_epoch = 0
resume_state = maybe_resume_training() if AUTO_RESUME else None
if resume_state is not None:
    start_epoch = resume_state['start_epoch']
    train_history = resume_state['train_history']
    val_history = resume_state['val_history']
    best_loss = resume_state['best_loss']
    if resume_state['mode'] == 'full':
        print(f"Resumed full training state from {resume_state['path']} at epoch {start_epoch}")
    else:
        print(f"Loaded legacy weights from {resume_state['path']} at epoch {start_epoch}; optimizer and history were reset")
else:
    print('Starting fresh training run')


def to_device(batch):
    moving = torch.from_numpy(batch['moving']).to(device)
    fixed = torch.from_numpy(batch['fixed']).to(device)
    moving_seg = torch.from_numpy(batch['moving_seg']).to(device)
    fixed_seg = torch.from_numpy(batch['fixed_seg']).to(device)
    return moving, fixed, moving_seg, fixed_seg


def run_step(batch, train=True):
    moving, fixed, moving_seg, fixed_seg = to_device(batch)

    if train:
        optimizer.zero_grad(set_to_none=True)
        flow = model(moving, fixed)
        smooth_loss = flow_gradient_loss_2d(flow)

        moving_seg_oh = seg_to_onehot(moving_seg)
        fixed_seg_oh = seg_to_onehot(fixed_seg)
        warped_seg_oh = spatial_transform_2d(moving_seg_oh, flow)
        dice_term = dice_loss(warped_seg_oh, fixed_seg_oh)

        moving_center = moving[:, CENTER_INDEX:CENTER_INDEX + 1]
        fixed_center = fixed[:, CENTER_INDEX:CENTER_INDEX + 1]
        warped_center = spatial_transform_2d(moving_center, flow)
        mi_loss = patchwise_mi(warped_center, fixed_center)

        total_loss = dice_term + MI_WEIGHT * mi_loss + SMOOTH_WEIGHT * smooth_loss
        if torch.isfinite(total_loss):
            total_loss.backward()
            optimizer.step()
    else:
        with torch.no_grad():
            flow = model(moving, fixed)
            smooth_loss = flow_gradient_loss_2d(flow)

            moving_seg_oh = seg_to_onehot(moving_seg)
            fixed_seg_oh = seg_to_onehot(fixed_seg)
            warped_seg_oh = spatial_transform_2d(moving_seg_oh, flow)
            dice_term = dice_loss(warped_seg_oh, fixed_seg_oh)

            moving_center = moving[:, CENTER_INDEX:CENTER_INDEX + 1]
            fixed_center = fixed[:, CENTER_INDEX:CENTER_INDEX + 1]
            warped_center = spatial_transform_2d(moving_center, flow)
            mi_loss = patchwise_mi(warped_center, fixed_center)

            total_loss = dice_term + MI_WEIGHT * mi_loss + SMOOTH_WEIGHT * smooth_loss

    flow_abs_mean = flow.detach().abs().mean().item()
    flow_std = flow.detach().std().item()

    return {
        'total': float(total_loss.detach().item()),
        'dice': float(dice_term.detach().item()),
        'mi_loss': float(mi_loss.detach().item()),
        'smooth': float(smooth_loss.detach().item()),
        'flow_abs_mean': float(flow_abs_mean),
        'flow_std': float(flow_std),
    }


def mean_metrics(metric_rows):
    keys = metric_rows[0].keys()
    return {key: float(np.mean([row[key] for row in metric_rows])) for key in keys}


Starting fresh training run


In [ ]:
latest_exists = os.path.exists(LATEST_TRAIN_STATE_PATH)
print('Resume status')
print('  auto_resume:', AUTO_RESUME)
print('  latest checkpoint:', LATEST_TRAIN_STATE_PATH)
print('  latest exists:', latest_exists)
if resume_state is not None:
    print('  mode:', resume_state['mode'])
    print('  source:', resume_state['path'])
    print('  start epoch:', start_epoch)
    print('  epochs target:', EPOCHS)
    print('  best val loss:', best_loss)
    if start_epoch >= EPOCHS:
        print('  action: no training left; notebook will skip the loop')
    else:
        print(f'  action: resume from epoch {start_epoch + 1}')
else:
    print('  action: start fresh training run')


Resume status
  auto_resume: False
  latest checkpoint: ../trained_weights/2p5d_pt_v2_checkpoints/2p5d_pt_v2_latest.pth
  latest exists: True
  action: start fresh training run


In [ ]:
if start_epoch >= EPOCHS:
    print(f'Checkpoint already reached epoch {start_epoch}; no training steps remain.')
else:
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_rows = []
        nan_steps = 0

        for _ in range(STEPS_PER_EPOCH):
            batch = next(train_gen)
            metrics = run_step(batch, train=True)
            if all(np.isfinite(v) for v in metrics.values()):
                train_rows.append(metrics)
            else:
                nan_steps += 1

        epoch_train = mean_metrics(train_rows) if train_rows else {key: float('nan') for key in train_history}
        for key, value in epoch_train.items():
            train_history[key].append(value)

        model.eval()
        val_rows = []
        with torch.no_grad():
            for batch in val_batches:
                val_rows.append(run_step(batch, train=False))
        epoch_val = mean_metrics(val_rows)
        for key, value in epoch_val.items():
            val_history[key].append(value)

        nan_str = f' ({nan_steps} NaN batches skipped)' if nan_steps else ''
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"train total={epoch_train['total']:.6f} dice={epoch_train['dice']:.6f} "
            f"mi={epoch_train['mi_loss']:.6f} smooth={epoch_train['smooth']:.6f} | "
            f"val total={epoch_val['total']:.6f} dice={epoch_val['dice']:.6f} "
            f"mi={epoch_val['mi_loss']:.6f} smooth={epoch_val['smooth']:.6f}{nan_str}"
        )

        if epoch_val['total'] < best_loss:
            best_loss = epoch_val['total']
            torch.save(model.state_dict(), BEST_MODEL_PATH)

        save_training_state(LATEST_TRAIN_STATE_PATH, epoch + 1, epoch_train, epoch_val)

print('\nBest val total loss:', best_loss)
print('Best model:', BEST_MODEL_PATH)
print('Latest training state:', LATEST_TRAIN_STATE_PATH)


In [ ]:
def compute_dice_per_label(warped_seg, fixed_seg, labels=SEG_LABELS):
    dice_scores = []
    for lbl in labels:
        w = (warped_seg == lbl).astype(np.float32)
        f = (fixed_seg == lbl).astype(np.float32)
        inter = float((w * f).sum())
        union = float(w.sum() + f.sum())
        dice_scores.append(1.0 if union == 0.0 else 2.0 * inter / union)
    return np.array(dice_scores, dtype=np.float32)


def mutual_information_np(a, b, bins=64, clip_range=(-1.0, 1.0)):
    hist_2d, _, _ = np.histogram2d(
        a.ravel(),
        b.ravel(),
        bins=bins,
        range=[clip_range, clip_range],
    )
    pxy = hist_2d / np.maximum(hist_2d.sum(), 1.0)
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    nz = pxy > 0
    return float((pxy[nz] * np.log(pxy[nz] / (px @ py)[nz])).sum())


def negative_jacobian_ratio_2d(flow_np):
    h, w, _ = flow_np.shape
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    deform = flow_np.copy()
    deform[..., 0] += xx
    deform[..., 1] += yy
    d00 = np.gradient(deform[..., 0], axis=0)
    d01 = np.gradient(deform[..., 0], axis=1)
    d10 = np.gradient(deform[..., 1], axis=0)
    d11 = np.gradient(deform[..., 1], axis=1)
    det = d00 * d11 - d01 * d10
    return float(np.mean(det <= 0))


def build_eval_pairs(n_subj, max_pairs=MAX_EVAL_PAIRS, seed=EVAL_PAIR_SEED):
    if n_subj < 2:
        return []
    rng = np.random.default_rng(seed)
    order = rng.permutation(n_subj)
    pairs = []
    for idx in range(0, len(order) - 1, 2):
        pairs.append((int(order[idx]), int(order[idx + 1])))
        if len(pairs) >= max_pairs:
            break
    return pairs


@torch.no_grad()
def infer_volume_orientation(model, moving_vol, fixed_vol, moving_seg, fixed_seg, orient_cfg, batch_size=EVAL_BATCH_SIZE):
    axis = orient_cfg['axis']
    target_hw = orient_cfg['target_hw']
    depth = min(moving_vol.shape[axis], fixed_vol.shape[axis], moving_seg.shape[axis], fixed_seg.shape[axis])
    slice_indices = list(range(WINDOW_RADIUS, depth - WINDOW_RADIUS))

    moving_center_slices = []
    fixed_center_slices = []
    warped_center_slices = []
    moving_seg_slices = []
    fixed_seg_slices = []
    warped_seg_slices = []
    flow_slices = []
    batch_times_ms = []

    for start in range(0, len(slice_indices), batch_size):
        batch_indices = slice_indices[start:start + batch_size]
        moving_stack_batch = []
        fixed_stack_batch = []
        moving_center_batch = []
        fixed_center_batch = []
        moving_seg_batch = []
        fixed_seg_batch = []

        for z in batch_indices:
            moving_stack = resize_stack(extract_stack(moving_vol, axis, z), target_hw)
            fixed_stack = resize_stack(extract_stack(fixed_vol, axis, z), target_hw)
            moving_center = resize_slice(extract_slice(moving_vol, axis, z), target_hw, is_seg=False)
            fixed_center = resize_slice(extract_slice(fixed_vol, axis, z), target_hw, is_seg=False)
            moving_seg_slice = resize_slice(extract_slice(moving_seg, axis, z), target_hw, is_seg=True).astype(np.int64)
            fixed_seg_slice = resize_slice(extract_slice(fixed_seg, axis, z), target_hw, is_seg=True).astype(np.int64)

            moving_stack_batch.append(moving_stack)
            fixed_stack_batch.append(fixed_stack)
            moving_center_batch.append(moving_center)
            fixed_center_batch.append(fixed_center)
            moving_seg_batch.append(moving_seg_slice)
            fixed_seg_batch.append(fixed_seg_slice)

        moving_t = torch.from_numpy(np.stack(moving_stack_batch).astype(np.float32)).to(device)
        fixed_t = torch.from_numpy(np.stack(fixed_stack_batch).astype(np.float32)).to(device)
        moving_center_t = torch.from_numpy(np.stack(moving_center_batch)[:, None].astype(np.float32)).to(device)
        fixed_center_t = torch.from_numpy(np.stack(fixed_center_batch)[:, None].astype(np.float32)).to(device)
        moving_seg_t = torch.from_numpy(np.stack(moving_seg_batch).astype(np.int64)).to(device)
        fixed_seg_t = torch.from_numpy(np.stack(fixed_seg_batch).astype(np.int64)).to(device)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        flow = model(moving_t, fixed_t)
        warped_center = spatial_transform_2d(moving_center_t, flow)
        warped_seg_oh = spatial_transform_2d(seg_to_onehot(moving_seg_t), flow)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        batch_times_ms.append((time.perf_counter() - t0) * 1000.0)

        moving_center_slices.append(moving_center_t[:, 0].cpu().numpy())
        fixed_center_slices.append(fixed_center_t[:, 0].cpu().numpy())
        warped_center_slices.append(warped_center[:, 0].cpu().numpy())
        moving_seg_slices.append(moving_seg_t.cpu().numpy())
        fixed_seg_slices.append(fixed_seg_t.cpu().numpy())
        warped_seg_slices.append(onehot_to_seg_numpy(warped_seg_oh))
        flow_slices.append(flow.cpu().numpy())

    moving_stack = np.concatenate(moving_center_slices, axis=0)
    fixed_stack = np.concatenate(fixed_center_slices, axis=0)
    warped_stack = np.concatenate(warped_center_slices, axis=0)
    moving_seg_stack = np.concatenate(moving_seg_slices, axis=0)
    fixed_seg_stack = np.concatenate(fixed_seg_slices, axis=0)
    warped_seg_stack = np.concatenate(warped_seg_slices, axis=0)
    flow_stack = np.concatenate(flow_slices, axis=0)

    dice_before = compute_dice_per_label(moving_seg_stack, fixed_seg_stack)
    dice_after = compute_dice_per_label(warped_seg_stack, fixed_seg_stack)
    jacobian = float(np.mean([
        negative_jacobian_ratio_2d(flow_stack[i].transpose(1, 2, 0))
        for i in range(flow_stack.shape[0])
    ]))

    summary = {
        'orientation': orient_cfg['name'],
        'axis': int(axis),
        'target_hw': list(target_hw),
        'slice_count': int(len(slice_indices)),
        'mean_batch_time_ms': float(np.mean(batch_times_ms)),
        'mi_before': mutual_information_np(moving_stack, fixed_stack),
        'mi_after': mutual_information_np(warped_stack, fixed_stack),
        'dice_before': float(dice_before.mean()),
        'dice_after': float(dice_after.mean()),
        'dice_before_per_label': {str(lbl): float(score) for lbl, score in zip(SEG_LABELS, dice_before)},
        'dice_after_per_label': {str(lbl): float(score) for lbl, score in zip(SEG_LABELS, dice_after)},
        'flow_min': float(flow_stack.min()),
        'flow_max': float(flow_stack.max()),
        'flow_mean': float(flow_stack.mean()),
        'flow_std': float(flow_stack.std()),
        'negative_jacobian_ratio_2d': jacobian,
    }

    return {
        'moving_stack': moving_stack,
        'fixed_stack': fixed_stack,
        'warped_stack': warped_stack,
        'moving_seg_stack': moving_seg_stack,
        'fixed_seg_stack': fixed_seg_stack,
        'warped_seg_stack': warped_seg_stack,
        'flow_stack': flow_stack,
        'summary': summary,
    }


def save_orientation_payload(base_dir, payload):
    os.makedirs(base_dir, exist_ok=True)
    np.save(os.path.join(base_dir, 'moving_stack.npy'), payload['moving_stack'])
    np.save(os.path.join(base_dir, 'fixed_stack.npy'), payload['fixed_stack'])
    np.save(os.path.join(base_dir, 'warped_stack.npy'), payload['warped_stack'])
    np.save(os.path.join(base_dir, 'moving_seg_stack.npy'), payload['moving_seg_stack'])
    np.save(os.path.join(base_dir, 'fixed_seg_stack.npy'), payload['fixed_seg_stack'])
    np.save(os.path.join(base_dir, 'warped_seg_stack.npy'), payload['warped_seg_stack'])
    np.save(os.path.join(base_dir, 'flow_stack.npy'), payload['flow_stack'])
    with open(os.path.join(base_dir, 'summary.json'), 'w') as f:
        json.dump(payload['summary'], f, indent=2)


In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

eval_pairs = build_eval_pairs(len(val_mr_vols), max_pairs=MAX_EVAL_PAIRS)
print('Eval pairs:', eval_pairs)

all_results = []
for pair_idx, (moving_idx, fixed_idx) in enumerate(eval_pairs):
    pair_dir = os.path.join(EVAL_DIR, f'pair_{pair_idx:02d}_m{moving_idx:03d}_f{fixed_idx:03d}')
    os.makedirs(pair_dir, exist_ok=True)

    pair_summary = {
        'pair_index': pair_idx,
        'moving_idx': moving_idx,
        'fixed_idx': fixed_idx,
        'orientations': {},
    }

    moving_vol = val_mr_vols[moving_idx]
    fixed_vol = val_ct_vols[fixed_idx]
    moving_seg = val_mr_segs[moving_idx]
    fixed_seg = val_ct_segs[fixed_idx]

    for orient_cfg in ORIENTATIONS:
        payload = infer_volume_orientation(model, moving_vol, fixed_vol, moving_seg, fixed_seg, orient_cfg)
        orient_dir = os.path.join(pair_dir, orient_cfg['name'])
        save_orientation_payload(orient_dir, payload)
        pair_summary['orientations'][orient_cfg['name']] = payload['summary']
        print(
            f"pair {pair_idx} | {orient_cfg['name']}: "
            f"dice {payload['summary']['dice_before']:.4f} -> {payload['summary']['dice_after']:.4f}, "
            f"mi {payload['summary']['mi_before']:.4f} -> {payload['summary']['mi_after']:.4f}"
        )

    with open(os.path.join(pair_dir, 'pair_summary.json'), 'w') as f:
        json.dump(pair_summary, f, indent=2)
    all_results.append(pair_summary)

with open(os.path.join(EVAL_DIR, 'summary.json'), 'w') as f:
    json.dump(all_results, f, indent=2)

print('Saved evaluation outputs to', EVAL_DIR)


Eval pairs: [(11, 0), (14, 5)]
pair 0 | axial: dice 0.5615 -> 0.7158, mi 0.5507 -> 0.6751
pair 0 | coronal: dice 0.5650 -> 0.7210, mi 0.5513 -> 0.6696
pair 0 | sagittal: dice 0.5610 -> 0.7115, mi 0.5492 -> 0.6712
pair 1 | axial: dice 0.6578 -> 0.7644, mi 0.5642 -> 0.6863
pair 1 | coronal: dice 0.6558 -> 0.7341, mi 0.5643 -> 0.6922
pair 1 | sagittal: dice 0.6501 -> 0.7381, mi 0.5611 -> 0.6803
Saved evaluation outputs to ./artifacts/results/train_2p5d_v2
